In [2]:
import tkinter as tk
from tkinter import messagebox
import pymysql

class DictionaryApp:
    def __init__(self, root):
        self.root = root
        self.root.title("Python 수업 용어 사전")
        self.root.geometry("400x550")
        
        # 접속할 DB 주소 (필요에 따라 수정하세요)
        self.db_host = "127.0.0.1" 
        self.connection = None
        self.suggestion_window = None # 자동완성 팝업 창 추적용
        
        # 로그인 화면 구성
        self.login_frame = tk.Frame(self.root)
        self.login_frame.pack(pady=50)
        
        tk.Label(self.login_frame, text="DB 계정 ID:").pack()
        self.ent_user = tk.Entry(self.login_frame)
        self.ent_user.pack()
        self.ent_user.bind('<Return>', lambda event: self.ent_pw.focus())
        
        tk.Label(self.login_frame, text="Password:").pack()
        self.ent_pw = tk.Entry(self.login_frame, show="*")
        self.ent_pw.pack()
        self.ent_pw.bind('<Return>', lambda event: self.connect_db())
        
        tk.Button(self.login_frame, text="DB 접속", command=self.connect_db).pack(pady=10)

    def connect_db(self):
        """MySQL 서버에 접속 시도"""
        user_id = self.ent_user.get()
        user_pw = self.ent_pw.get()
        
        try:
            self.connection = pymysql.connect(
                host=self.db_host,
                user=user_id,
                password=user_pw,
                db='class_dict_db',
                charset='utf8mb4',
                cursorclass=pymysql.cursors.DictCursor
            )
            messagebox.showinfo("성공", "데이터베이스에 연결되었습니다.")
            self.show_search_ui()
        except Exception as e:
            messagebox.showerror("오류", f"접속 실패: {e}")

    def show_search_ui(self):
        """로그인 성공 후 검색 및 등록 UI 화면"""
        self.login_frame.destroy() 
        
        self.search_frame = tk.Frame(self.root)
        self.search_frame.pack(pady=20, fill="both", expand=True)
        
        tk.Label(self.search_frame, text="검색할 용어를 입력하세요:").pack()
        
        # 검색창 구성 및 이벤트 바인딩
        self.ent_search = tk.Entry(self.search_frame, width=30)
        self.ent_search.pack(pady=5)
        # 키를 뗄 때마다 실시간으로 자동완성 목록 확인
        self.ent_search.bind('<KeyRelease>', self.on_key_release)
        self.ent_search.bind('<Return>', lambda e: self.search_word())
        
        tk.Button(self.search_frame, text="상세 검색", command=self.search_word).pack(pady=5)
        
        self.txt_result = tk.Text(self.search_frame, height=12, width=45)
        self.txt_result.pack(pady=10)

        # 새 단어 등록 버튼
        tk.Button(self.search_frame, text="새 단어 등록하기", 
                  command=self.open_add_window, bg="#e1e1e1").pack(pady=10)

    def on_key_release(self, event):
        """키 입력 시 실시간으로 DB에서 비슷한 단어를 찾아 팝업 생성"""
        search_text = self.ent_search.get().strip()
        
        # 팝업 창 초기화
        if self.suggestion_window:
            self.suggestion_window.destroy()
            self.suggestion_window = None
            
        if not search_text:
            return

        try:
            with self.connection.cursor() as cursor:
                # 입력값이 포함된 단어 최대 10개 추출
                sql = "SELECT word FROM terms WHERE word LIKE %s LIMIT 10"
                cursor.execute(sql, (f"%{search_text}%",))
                results = cursor.fetchall()
                
                if results:
                    self.show_suggestion_popup(results)
        except Exception as e:
            print(f"목록 갱신 오류: {e}")

    def show_suggestion_popup(self, results):
        """검색창 아래에 단어 목록 팝업 생성"""
        self.suggestion_window = tk.Toplevel(self.root)
        self.suggestion_window.wm_overrideredirect(True) # 타이틀바 제거
        
        # 검색창 바로 아래 위치 계산
        x = self.ent_search.winfo_rootx()
        y = self.ent_search.winfo_rooty() + self.ent_search.winfo_height()
        self.suggestion_window.geometry(f"215x150+{x}+{y}")
        
        listbox = tk.Listbox(self.suggestion_window, width=30)
        listbox.pack(fill="both", expand=True)
        
        for item in results:
            listbox.insert(tk.END, item['word'])
            
        # 목록 클릭 시 선택 이벤트
        listbox.bind("<<ListboxSelect>>", lambda e: self.select_from_list(listbox))

    def select_from_list(self, listbox):
        """리스트에서 단어 선택 시 검색창에 넣고 즉시 검색"""
        if not listbox.curselection():
            return
        selected_word = listbox.get(listbox.curselection())
        self.ent_search.delete(0, tk.END)
        self.ent_search.insert(0, selected_word)
        self.suggestion_window.destroy()
        self.suggestion_window = None
        self.search_word()

    def search_word(self):
        """DB에서 단어의 정확한 뜻 검색"""
        word = self.ent_search.get().strip()
        if not word:
            return

        try:
            with self.connection.cursor() as cursor:
                sql = "SELECT definition FROM terms WHERE word = %s"
                cursor.execute(sql, (word,))
                result = cursor.fetchone()
                
                self.txt_result.delete(1.0, tk.END)
                if result:
                    self.txt_result.insert(tk.END, f"[{word}]의 뜻:\n\n{result['definition']}")
                else:
                    self.txt_result.insert(tk.END, "정확한 검색 결과가 없습니다.")
        except Exception as e:
            messagebox.showerror("오류", f"조회 중 오류 발생: {e}")

    def open_add_window(self):
        """새 단어를 입력하는 별도의 창 생성"""
        self.add_win = tk.Toplevel(self.root)
        self.add_win.title("새 용어 등록")
        self.add_win.geometry("350x400")
        
        tk.Label(self.add_win, text="단어명:").pack(pady=10)
        self.ent_new_word = tk.Entry(self.add_win, width=30)
        self.ent_new_word.pack()
        
        tk.Label(self.add_win, text="단어 내용(뜻):").pack(pady=10)
        self.txt_new_def = tk.Text(self.add_win, height=10, width=35)
        self.txt_new_def.pack()
        
        tk.Button(self.add_win, text="데이터베이스에 기록", 
                  command=self.save_to_db, bg="#4CAF50", fg="white").pack(pady=20)

    def save_to_db(self):
        """새 단어 등록 및 중복 체크"""
        new_word = self.ent_new_word.get().strip()
        new_def = self.txt_new_def.get(1.0, tk.END).strip()
        
        if not new_word or not new_def:
            messagebox.showwarning("입력 누락", "단어명과 내용을 모두 입력해주세요.")
            return

        try:
            with self.connection.cursor() as cursor:
                # 1. 중복 확인
                check_sql = "SELECT word FROM terms WHERE word = %s"
                cursor.execute(check_sql, (new_word,))
                if cursor.fetchone():
                    messagebox.showwarning("중복 오류", f"'{new_word}'은(는) 이미 등록된 단어입니다.")
                    return

                # 2. 데이터 삽입
                sql = "INSERT INTO terms (word, definition) VALUES (%s, %s)"
                cursor.execute(sql, (new_word, new_def))
                self.connection.commit() 
                
            messagebox.showinfo("성공", f"'{new_word}' 단어가 등록되었습니다.")
            self.add_win.destroy()
        except Exception as e:
            if self.connection:
                self.connection.rollback()
            messagebox.showerror("오류", f"데이터 저장 중 오류 발생: {e}")

if __name__ == "__main__":
    root = tk.Tk()
    app = DictionaryApp(root)
    root.mainloop()